## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obtained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer.

### Model Training

For this exercise, it is necessary to have a model registered in MLFlow. Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
import mlflow
from mlflow.models import infer_signature

In [ ]:
ONE_HOT_ENCODE_COLUMNS = ['Geography', 'Gender']
encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform='pandas')

def transform(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, errors='ignore').copy()
    df2['HasBalance'] = (df2['Balance'] > 0).astype(int)
    df2 = df2.dropna().reset_index(drop=True)

    age_cap = df2['Age'].quantile(0.99)
    df2['Age'] = df2['Age'].clip(upper=age_cap)

    Q1 = df2['CreditScore'].quantile(0.10)
    Q3 = df2['CreditScore'].quantile(0.90)
    df2['CreditScore'] = df2['CreditScore'].clip(lower=Q1 - 1.5*(Q3-Q1), upper=Q3 + 1.5*(Q3-Q1))

    encoder.fit(df2[ONE_HOT_ENCODE_COLUMNS])
    encoded_df = encoder.transform(df2[ONE_HOT_ENCODE_COLUMNS])
    df2 = df2.drop(columns=ONE_HOT_ENCODE_COLUMNS)
    df2 = pd.concat([df2, encoded_df], axis=1)
    return df2

In [ ]:
df = pd.read_csv('Churn_Modelling_train_test.csv')
processed_df = transform(df)

joblib.dump(encoder, 'encoder.pkl')
print('Encoder saved')

In [ ]:
params = {
    'max_depth': 6,
    'min_samples_split': 9,
    'min_samples_leaf': 3,
    'class_weight': 'balanced',
    'random_state': 42
}

X = processed_df.drop('Exited', axis=1)
y = processed_df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(**params)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f'Test AUC: {auc:.4f}')

In [ ]:
mlflow.set_tracking_uri(uri='http://127.0.0.1:8080')
mlflow.set_experiment('Practice Experiment - Santiago')

with mlflow.start_run() as run:
    mlflow.log_params(params)
    mlflow.log_metric('test_auc', auc)
    mlflow.set_tag('Training Info', 'Session 3 - Decision Tree for deployment exercise')
    signature = infer_signature(X_train, model.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path='decision_tree_model',
        signature=signature,
        input_example=X_train,
        registered_model_name='DecisionTreeChurnModel'
    )
    model_uri = f'runs:/{run.info.run_id}/decision_tree_model'
    print(f'Model URI: {model_uri}')

### Inference

In this part, batch and online inference are implemented using the registered model.

##### Batch Inference

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

class ModelBatchPredictor:

    def _apply_transformation(self, df: pd.DataFrame) -> pd.DataFrame:
        df = transform(df.copy())
        return df.drop(columns=['Exited'], errors='ignore')

    def batch_inference(self, model_uri: str, input: pd.DataFrame) -> list:
        model = mlflow.pyfunc.load_model(model_uri)
        input_transformed = self._apply_transformation(input)
        return model.predict(input_transformed)

In [ ]:
validation_df = pd.read_csv('Churn_Modelling_val.csv')
y_validation = validation_df['Exited']

predictor = ModelBatchPredictor()
predictions = predictor.batch_inference(model_uri=model_uri, input=validation_df)

print(confusion_matrix(y_validation, predictions))
print(f'Accuracy: {accuracy_score(y_validation, predictions):.4f}')

##### Online Inference

To set up the local inference server:
1. Open a new terminal
2. Run: `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080`
3. Run: `mlflow models serve -m <model_uri> -p 5001 --no-conda`

In [ ]:
import requests
import json

In [ ]:
class MLflowInferenceClient:

    def __init__(self, host='http://127.0.0.1', port=5001):
        self.url = f'{host}:{port}/invocations'

    def predict_json(self, json_data: dict) -> requests.Response:
        headers = {'Content-Type': 'application/json'}
        return requests.post(self.url, headers=headers, json=json_data)

    def predict_pandas(self, input_df: pd.DataFrame) -> requests.Response:
        headers = {'Content-Type': 'application/json'}
        payload = {'dataframe_split': input_df.to_dict(orient='split')}
        return requests.post(self.url, headers=headers, data=json.dumps(payload))

client = MLflowInferenceClient()

In [ ]:
sample = X_test.head(1).to_dict(orient='split')
json_payload = {'dataframe_split': sample}

response = client.predict_json(json_payload)
print(response.content)

In [ ]:
response_pandas = client.predict_pandas(X_test.head(1))
print(response_pandas.content)